# Python Threading and Server Examples


## 1. Simple Thread Creation


In [1]:
import threading

# Global thread ID counter and a lock for thread - safe updates
tid = 1

tid_lock = threading.Lock()


def print_thread_id():
    global tid
    with tid_lock:
        current_id = tid
        tid += 1
        print(f"This is Thread: {current_id}")


def main():
    # Create two threads
    thread1 = threading.Thread(target=print_thread_id)
    thread2 = threading.Thread(target=print_thread_id)

    # Start the threads
    thread1.start()
    thread2.start()

    # Wait for both threads to complete
    thread1.join()
    thread2.join()


if __name__ == "__main__":
    main()

This is Thread: 1
This is Thread: 2


## 2. Passing Arguments to a Thread


In [2]:
import threading


def print_thread_id(message):
    print(message)


def main():
    message1 = "Thread 1"
    message2 = "Thread 2"

    # Create threads and pass arguments
    thread1 = threading.Thread(target=print_thread_id, args=(message1,))
    thread2 = threading.Thread(target=print_thread_id, args=(message2,))

    # Start the threads
    thread1.start()
    thread2.start()

    # Wait for both threads to finish
    thread1.join()
    thread2.join()


if __name__ == "__main__":
    main()

Thread 1
Thread 2


## 3. Multithreaded Group Chat Server


### Chat Server (chat server threaded.py)


In [ ]:
import socket
import threading

clients = []
clients_lock = threading.Lock()


def broadcast(message, sender_conn):
    """Sends a message to all clients except the sender."""
    with clients_lock:
        for client_conn in clients:
            if client_conn != sender_conn:
                try:
                    client_conn.sendall(message)
                except socket.error:
                    clients.remove(client_conn)


def handle_client(conn, addr):
    """Handles a single client, receiving and broadcasting messages."""
    print(f"[NEW CONNECTION] {addr} connected.")
    with clients_lock:
        clients.append(conn)

    try:
        while True:
            message = conn.recv(1024)
            if not message:
                break

            broadcast_message = f"[{addr[0]}:{addr[1]}] {message.decode()}".encode()
            broadcast(broadcast_message, conn)
    finally:
        with clients_lock:
            clients.remove(conn)
        print(f"[DISCONNECTED] {addr} disconnected.")
        conn.close()


def main():
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.bind(("127.0.0.1", 2001))
    server.listen()

    print("[*] Chat Server listening on 127.0.0.1:2001")
    while True:
        conn, addr = server.accept()
        thread = threading.Thread(target=handle_client, args=(conn, addr))
        thread.start()


if __name__ == "__main__":
    main()

### Chat Client (chat client threaded.py)


In [ ]:
import socket
import threading


def receive_message(client_socket):
    """Continuously receives messages from the server."""
    while True:
        try:
            message = client_socket.recv(1024).decode()
            if not message:
                print("\nDisconnected from server.")
                break
            print(message)
        except:
            print("\nConnection to the server was lost.")
            break


def main():
    client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    client.connect(("127.0.0.1", 2001))
    print("Connected to the Chat Server! Type and press ENTER to send.")

    receive_thread = threading.Thread(target=receive_message, args=(client,))
    receive_thread.daemon = True
    receive_thread.start()

    while True:
        try:
            message = input()
            client.sendall(message.encode())
        except (KeyboardInterrupt, EOFError):
            break

    client.close()


if __name__ == "__main__":
    main()